# Adaptive Loop — RunPod Notebook

Phase 3: Per-identity adaptive LoRA scale feedback loop.

**İlk çalıştırma sırasını:**
1. `Cell A` — repo klonla (sadece ilk sefer)
2. `Cell B` — LoRA dosyalarını indir (sadece ilk sefer)
3. Diğer tüm cell'leri sırayla çalıştır

## Cell A — Repo'yu klonla (ilk sefer)

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/LoRA-scale-adaptive-feedback.git"
REPO_DIR = "/workspace/LoRA-scale-adaptive-feedback"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already at {REPO_DIR}")
    !cd {REPO_DIR} && git pull

## Cell B — LoRA dosyalarını Google Drive klasöründen indir (ilk sefer)

Klasör 'Anyone with the link can view' olmalı.

In [ ]:
!pip install -q gdown
import os
os.makedirs(f"{REPO_DIR}/data/loras", exist_ok=True)

DRIVE_FOLDER_ID = "1L6NA_AT5HGSNOcKjw7gECbPKnaN86Ix5"

!gdown --folder "https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}" -O {REPO_DIR}/data/loras/

# gdown may create a subfolder — flatten if needed
import glob, shutil
for f in glob.glob(f"{REPO_DIR}/data/loras/**/*.safetensors", recursive=True):
    target = os.path.join(f"{REPO_DIR}/data/loras", os.path.basename(f))
    if f != target:
        shutil.move(f, target)

!ls -lh {REPO_DIR}/data/loras/

## Setup caches + dependencies

In [ ]:
import os
os.environ["HF_HOME"] = "/workspace/cache/huggingface"
os.environ["INSIGHTFACE_HOME"] = "/workspace/cache/insightface"
os.makedirs("/workspace/cache/huggingface", exist_ok=True)
os.makedirs("/workspace/cache/insightface", exist_ok=True)
print("Caches set up")

In [ ]:
!pip install -q diffusers==0.30.3 transformers==4.44.2 peft controlnet-aux insightface onnxruntime-gpu

In [ ]:
os.chdir("/workspace/LoRA-scale-adaptive-feedback")
import sys
sys.path.insert(0, "src")

print("LoRAs:")
!ls -lh data/loras/
print("\nReference faces:")
!ls data/reference_faces/
print("\nPose images:")
!ls data/pose_images/

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Build pipeline + scorers

In [ ]:
from pipeline import load_identities, build_pipeline
from scorer import FaceScorer
from pose_scorer import PoseScorer
from PIL import Image

identities = load_identities()
pipe = build_pipeline(identities)

pose_image = Image.open("data/pose_images/two_person_pose.png").convert("RGB")

face_scorer = FaceScorer(reference_dir="data/reference_faces", device="cuda")
pose_scorer = PoseScorer()
target_keypoints = pose_scorer.extract_and_cache_target(pose_image)

names = list(identities.keys())
identity_regions = {
    names[0]: (0,   0, 512,  1024),
    names[1]: (512, 0, 1024, 1024),
}

print("Ready.")

## Single adaptive run (sanity check)

In [ ]:
from adaptive_loop import multi_lora_adaptive_generate

result = multi_lora_adaptive_generate(
    pipe, identities, pose_image,
    face_scorer, pose_scorer, target_keypoints, identity_regions,
    ctrl_scale=0.7,
    id_threshold=0.40,
    pose_threshold=0.50,
    alpha_init=0.5,
    delta_up=0.15,
    delta_down=0.10,
    max_iters=5,
    seed=42,
    use_regional_attention=True,
)

print(f"\nStatus: {result.status}")
print(f"Iterations: {result.total_iterations}")
print(f"Final alphas: {result.final_alphas}")
print(f"Final arcface: {result.final_arcface}")
print(f"Final pose:    {result.final_pose}")

display(result.image.resize((512, 512)))

## Full experiment — multiple seeds

In [ ]:
from adaptive_loop import run_adaptive_experiment
import pandas as pd

SEEDS = [42, 123, 777]

records = run_adaptive_experiment(
    pipe, identities, pose_image,
    face_scorer, pose_scorer, target_keypoints, identity_regions,
    seeds=SEEDS,
    output_dir="data/results/adaptive",
    save_images=True,
    use_regional_attention=True,
    ctrl_scale=0.7,
    id_threshold=0.40,
    pose_threshold=0.50,
    alpha_init=0.5,
    delta_up=0.15,
    delta_down=0.10,
    max_iters=5,
)

df_adaptive = pd.DataFrame(records)
df_adaptive.to_csv("data/results/adaptive/adaptive_runs.csv", index=False)
df_adaptive

## Ablation — with vs without regional attention

In [ ]:
records_no_mask = run_adaptive_experiment(
    pipe, identities, pose_image,
    face_scorer, pose_scorer, target_keypoints, identity_regions,
    seeds=SEEDS,
    output_dir="data/results/adaptive_no_mask",
    save_images=True,
    use_regional_attention=False,
    ctrl_scale=0.7,
    id_threshold=0.40,
    pose_threshold=0.50,
    alpha_init=0.5,
    delta_up=0.15,
    delta_down=0.10,
    max_iters=5,
)

df_no_mask = pd.DataFrame(records_no_mask)
df_no_mask.to_csv("data/results/adaptive_no_mask/adaptive_runs.csv", index=False)
df_no_mask